In [ ]:
from base64 import b64encode
from html import escape
from pathlib import Path
import random

import pandas as pd
from IPython.display import HTML, display
from inspect_ai.log import read_eval_log


In [ ]:
eval_dir = Path("../outputs/first_epoch/image_gen")
n_sample = 3
random_seed = 69

# Prepare data

In [ ]:
def score_for(sample):
    scores = getattr(sample, "scores", None)
    return next(iter(scores.values())) if scores else sample.score


def is_correct(sample):
    score = score_for(sample)
    return score is not None and score.value in (1, True, "C")


def image_path_for_sample(model_path, sample):
    matches = sorted((model_path / "generated_images").glob(f"generated_{sample.id}_*.png"))
    return str(matches[0]) if matches else None


def image_text(sample):
    if len(sample.messages) < 2 or not isinstance(sample.messages[1].content, list):
        return ""
    return chr(10).join(
        part.text for part in sample.messages[1].content
        if getattr(part, "type", None) == "text"
    )


def load_model_rows(model_path):
    eval_file = next(model_path.glob("*.eval"))
    log = read_eval_log(str(eval_file))
    valid = [s for s in log.samples if score_for(s) is not None and s.error is None]
    correct = [s for s in valid if is_correct(s)]
    incorrect = [s for s in valid if not is_correct(s)]

    rng = random.Random(f"{random_seed}-{model_path.name}")
    selected = (
        [(s, True) for s in rng.sample(correct, min(n_sample, len(correct)))] +
        [(s, False) for s in rng.sample(incorrect, min(n_sample, len(incorrect)))]
    )

    rows = []
    for sample, verifier_correct in selected:
        score = score_for(sample)
        rows.append({
            "model": model_path.name.rsplit("_", 2)[0],
            "run": model_path.name,
            "id": sample.id,
            "question": sample.input,
            "image_path": image_path_for_sample(model_path, sample),
            "target": sample.target,
            "verifier_score": "C" if verifier_correct else "I",
            "verifier_correct": verifier_correct,
            "verifier_explanation": score.explanation,
            "generation_note": image_text(sample),
        })
    return rows


In [ ]:
rows = []
for model_path in sorted(p for p in eval_dir.iterdir() if p.is_dir()):
    try:
        model_rows = load_model_rows(model_path)
        rows.extend(model_rows)
        counts = pd.Series([r["verifier_score"] for r in model_rows]).value_counts().to_dict()
        print(f"{model_path.name}: {counts}")
    except Exception as exc:
        print(f"Skipping {model_path.name}: {exc}")

results_df = pd.DataFrame(rows)
results_df = results_df.sample(frac=1, random_state=random_seed).reset_index(drop=True)
results_df[["model", "id", "image_path"]]


# Human grading

In [ ]:
from base64 import b64encode
from html import escape
from IPython.display import HTML, display


def image_html(path, width=520):
    path = Path(path)
    if not path.exists():
        return f"<b>Missing image:</b> {escape(str(path))}"
    data = b64encode(path.read_bytes()).decode("ascii")
    return f'''
    <div>
      <img src="data:image/png;base64,{data}" style="max-width:{width}px; height:auto; border:1px solid #ccc;">
      <div style="font-size:12px; color:#777; margin-top:4px;">{escape(str(path))}</div>
    </div>
    '''


def show_case(row):
    print(f"Model: {row.model} | sample id: {row.id}")
    print()
    print("Question:")
    print(row.question)
    print()
    print("Model generation:")
    display(HTML(image_html(row.image_path)))
    print()
    print("Correctness criteria:")
    print(row.target)


for idx, row in results_df.iterrows():
    print(f"Case {idx + 1} of {len(results_df)}")
    show_case(row)
    import time; time.sleep(0.5)  # Small delay to ensure the image is rendered before asking for input

    answer = ""
    while answer.lower() not in ("y", "n"):
        answer = input("Did the model respond correctly? [y/n] ")

    results_df.loc[idx, "human_score"] = "C" if answer.lower() == "y" else "I"
    results_df.loc[idx, "human_correct"] = answer.lower() == "y"
    print()
    print("-" * 80)
    print()


# Compare human and verifier

In [ ]:
graded = results_df.dropna(subset=["human_score"]).copy()
graded["human_matches_verifier"] = graded.human_score == graded.verifier_score

summary = pd.DataFrame({
    "n": [len(graded)],
    "agreement": [graded.human_matches_verifier.mean()],
})
summary


In [ ]:
pd.crosstab(
    graded.human_score,
    graded.verifier_score,
    rownames=["human"],
    colnames=["verifier"],
    dropna=False,
)


In [ ]:
disagreements = graded.loc[
    ~graded.human_matches_verifier,
    ["model", "id", "question", "image_path", "human_score", "verifier_score", "verifier_explanation"],
]
disagreements


# Export

In [ ]:
out_path = Path("../data/validation/img_output_valid_2.csv")
results_df.to_csv(out_path, index=False)
out_path
